In [1]:
!pip install -U bitsandbytes>=0.46.1 -q
!pip install -U transformers -q
!pip install google-colab -q

In [42]:

import torch
import time
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

PROMPT = """
Explain why transformers consume a lot of memory.
"""

MAX_NEW_TOKENS = 120

TEMPERATURE = 0.8
TOP_P = 0.95
TOP_K = 50

USE_4BIT = True

# ============================================================
# GPU
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("\n==============================")
print("DEVICE")
print("==============================")

print(DEVICE)

if DEVICE == "cuda":

    print(torch.cuda.get_device_name(0))

    torch.backends.cuda.matmul.allow_tf32 = True

    torch.set_float32_matmul_precision("high")

# ============================================================
# QUANTIZATION
# ============================================================

bnb_config = None

if USE_4BIT:

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=False
    )

# ============================================================
# TOKENIZER
# ============================================================

print("\n==============================")
print("TOKENIZER")
print("==============================")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:

    tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# MODEL
# ============================================================

print("\n==============================")
print("MODEL")
print("==============================")

start_load = time.time()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

load_time = time.time() - start_load

print(f"\nLoad Time: {load_time:.2f}s")

# ============================================================
# INPUT
# ============================================================

inputs = tokenizer(
    PROMPT,
    return_tensors="pt"
)

input_ids = inputs["input_ids"].to(DEVICE)

attention_mask = inputs["attention_mask"].to(DEVICE)

# ============================================================
# GENERATION
# ============================================================

print("\n==============================")
print("GENERATING")
print("==============================")

start_generation = time.time()

with torch.inference_mode():

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=1.1,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

generation_time = time.time() - start_generation

# ============================================================
# DECODE
# ============================================================

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# ============================================================
# RESULTS
# ============================================================

print("\n==============================")
print("GENERATED TEXT")
print("==============================\n")

print(generated_text)

print("\n==============================")
print("BENCHMARK")
print("==============================\n")

print(f"Generation Time: {generation_time:.2f}s")

if DEVICE == "cuda":

    peak_memory = torch.cuda.max_memory_allocated() / 1024**3

    print(f"Peak GPU Memory: {peak_memory:.2f} GB")



DEVICE
cuda
Tesla T4

TOKENIZER

MODEL


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Load Time: 5.47s

GENERATING


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



GENERATED TEXT


Explain why transformers consume a lot of memory.

Step 4: Memory Usage

Memory usage in a transformer is dependent on the model size and the number of layers. When the model is small, there is less memory needed to store the weights and biases. However, as the model grows, it needs more memory, and this can be attributed to the number of parameters and the added weight matrices. This process continues until the memory reaches a maximum limit that may occur during training or evaluation phase.

For instance, when using a single-layer transformer with the default size (512 tokens) with a batch

BENCHMARK

Generation Time: 5.61s
Peak GPU Memory: 0.87 GB
